# TESSERA — Downstream Land Cover Classification

This notebook trains a scikit-learn classifier on TESSERA embeddings and compares it against a baseline trained on raw Sentinel-2 spectral features.

The key insight: TESSERA embeddings encode **time-series patterns** (seasonal vegetation cycles, flood dynamics, phenological timing) that are invisible in single-image spectral features. This advantage is most pronounced for tasks like:
- Crop type mapping (different crops have different phenological signatures)
- Deciduous vs. evergreen forest discrimination
- Wetland and flooded area detection

## References
- GeoTessera: https://github.com/ucam-eo/geotessera

## 1. Load Embeddings and Create a Labeled Dataset

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import os

# Load saved embeddings (from 1-fetching_embeddings.ipynb)
if os.path.exists("tessera_embeddings.zarr"):
    ds = xr.open_zarr("tessera_embeddings.zarr")
    emb = ds["embedding"].values  # [lat, lon, feature]
    print(f"Loaded embeddings: {emb.shape}")
else:
    print("tessera_embeddings.zarr not found. Run 1-fetching_embeddings.ipynb first.")
    print("Generating simulated embeddings for demonstration.")

    n_lat, n_lon, embed_dim = 100, 150, 128
    emb = np.random.randn(n_lat, n_lon, embed_dim).astype(np.float32)
    emb[60:, :, :] += 2.0

## 2. Simulate Land Cover Labels

In practice, obtain labels from:
- CORINE Land Cover / ESA WorldCover for European regions
- National land cover datasets (NLCD, CLC, etc.)
- Manually digitized samples

In [ ]:
class_names = ["Water", "Forest", "Cropland", "Urban", "Grassland"]
n_classes = len(class_names)
n_lat, n_lon, embed_dim = emb.shape

# Simulate spatially coherent labels
labels_2d = np.zeros((n_lat, n_lon), dtype=int)
labels_2d[:15, :] = 0       # water (top strip)
labels_2d[15:40, :60] = 1   # forest (left)
labels_2d[15:40, 60:] = 2   # cropland (right)
labels_2d[40:70, :] = 4     # grassland (middle)
labels_2d[70:, :] = 3       # urban (bottom)

# Make embedding cluster around class labels
class_offsets = np.array([
    [-3, -3, 0, 0],   # water: low in dims 0-1
    [3, 0, 3, 0],     # forest: high in dim 0, 2
    [0, 3, -3, 0],    # cropland: high dim 1, low dim 2
    [2, 2, 2, 2],     # urban: high all
    [0, -2, 2, -2],   # grassland: mixed
], dtype=np.float32)

for cls_idx in range(n_classes):
    mask = labels_2d == cls_idx
    offset = np.zeros(embed_dim, dtype=np.float32)
    offset[:4] = class_offsets[cls_idx]
    emb[mask] += offset

# Flatten to 2D
X = emb.reshape(-1, embed_dim)
y = labels_2d.reshape(-1)

print(f"Feature matrix: {X.shape}")
print(f"Labels: {y.shape}")
for cls_idx, cls_name in enumerate(class_names):
    print(f"  Class {cls_idx} ({cls_name}): {np.sum(y==cls_idx):,} pixels")

## 3. Simulate a Baseline: Raw Sentinel-2 Spectral Features

In [ ]:
# Simulate raw spectral features (6 Sentinel-2 bands, single date)
# These lack temporal information — the key advantage of TESSERA
spectral_signatures = np.array([
    [0.02, 0.03, 0.025, 0.015, 0.01, 0.008],   # water
    [0.03, 0.06, 0.04,  0.40,  0.09, 0.05],    # forest
    [0.04, 0.09, 0.07,  0.35,  0.18, 0.10],    # cropland
    [0.09, 0.10, 0.11,  0.12,  0.13, 0.12],    # urban
    [0.05, 0.10, 0.08,  0.30,  0.20, 0.12],    # grassland
], dtype=np.float32)

X_spectral = spectral_signatures[y] + np.random.normal(0, 0.02, (len(y), 6)).astype(np.float32)
print(f"Spectral features shape: {X_spectral.shape}")

## 4. Train and Compare Classifiers

In [ ]:
X_train_emb, X_test_emb, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train_spec, X_test_spec, _, _ = train_test_split(X_spectral, y, test_size=0.3, random_state=42, stratify=y)

scaler_emb = StandardScaler().fit(X_train_emb)
scaler_spec = StandardScaler().fit(X_train_spec)

classifiers = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

results = {}

for clf_name, clf in classifiers.items():
    # Train on TESSERA embeddings
    clf.fit(scaler_emb.transform(X_train_emb), y_train)
    acc_emb = clf.score(scaler_emb.transform(X_test_emb), y_test)

    # Train on spectral features
    from sklearn.ensemble import RandomForestClassifier as RF
    clf_spec = RF(n_estimators=100, random_state=42, n_jobs=-1)
    clf_spec.fit(scaler_spec.transform(X_train_spec), y_train)
    acc_spec = clf_spec.score(scaler_spec.transform(X_test_spec), y_test)

    results[clf_name] = {"tessera": acc_emb, "spectral": acc_spec}
    print(f"{clf_name}:")
    print(f"  TESSERA embeddings:    {acc_emb*100:.1f}%")
    print(f"  Sentinel-2 spectral:   {acc_spec*100:.1f}%")
    print(f"  Improvement: +{(acc_emb - acc_spec)*100:.1f}pp")

## 5. Confusion Matrix

In [ ]:
clf_final = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf_final.fit(scaler_emb.transform(X_train_emb), y_train)
y_pred = clf_final.predict(scaler_emb.transform(X_test_emb))

fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, colorbar=True, cmap="Blues")
ax.set_title("TESSERA + Random Forest — Confusion Matrix")
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

## 6. Predicted Map

In [ ]:
y_pred_all = clf_final.predict(scaler_emb.transform(X))
pred_map = y_pred_all.reshape(n_lat, n_lon)
true_map = y.reshape(n_lat, n_lon)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = axes[0].imshow(true_map, cmap="tab10", vmin=0, vmax=n_classes-1)
axes[0].set_title("Ground truth")
axes[0].axis("off")

im1 = axes[1].imshow(pred_map, cmap="tab10", vmin=0, vmax=n_classes-1)
axes[1].set_title("TESSERA + RF prediction")
axes[1].axis("off")

cbar = plt.colorbar(im1, ax=axes, orientation="vertical", fraction=0.02)
cbar.set_ticks(np.arange(n_classes))
cbar.set_ticklabels(class_names)

plt.suptitle("Land Cover Classification with TESSERA Embeddings")
plt.tight_layout()
plt.show()